# 实验三 · π 估算：数据竞争与锁粒度

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 实验二的并行之所以无需同步，前提是各线程写入互不重叠的区间。本实验刻意打破这一前提：多个线程累加到**同一个**变量，从而引出本章第一个真正的并发缺陷——**数据竞争**。
> 2. 实验流程为：制造一个算错的并行程序 → 分析其机器级成因 → 用互斥量修正 → 比较两种加锁粒度的性能差异，最终导出全章最重要的一条工程规则。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 数据竞争是否显现取决于线程调度，与核心数、系统负载均有关。请尽量在**华为鲲鹏多核处理器**上运行。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 陈述**数据竞争**的严格定义，并据以判断一段并行代码是否需要同步
- 从机器指令层面解释竞态的成因，说明为何一行 C 语句不等于一个原子操作
- 说明编译器优化如何**放大**数据竞争的后果，并用 `objdump` 加以验证
- 准确识别程序中的**临界区**，使用 `pthread_mutex_t` 予以保护
- 对比细粒度锁与粗粒度锁的性能，解释二者差距的来源
- 运用**最小化临界区**这一规则改写并行归约代码

## 🗺️ 学习路径

1. **准备阶段**：理解 Gregory-Leibniz 级数的并行分解，注意符号项带来的下标奇偶性问题
2. **理论分析**：从读—改—写三步出发，推导丢失更新的成因，给出数据竞争的严格定义
3. **版本一（竞态）**：不加任何同步，观察程序算出错误结果
   → 用 `objdump` 查明编译器对未同步共享写做了什么
4. **版本二（细粒度锁）**：每次迭代加锁，结果正确但性能急剧劣化
   → 认识「加锁本身不慢，慢的是加错了位置」
5. **版本三（粗粒度锁）**：局部累加后仅加锁一次，兼顾正确与高效
   → 导出**最小化临界区**规则

## 1. 背景与动机

实验二的并行版本没有使用任何锁，却完全正确。其前提是：`A`、`x` 只读，而 `y` 虽被写入，但**每个线程写的是互不重叠的区间**。

本实验打破这一前提。π 的估算是一个**归约**（reduction）问题：各线程分头计算部分和，最终必须汇总到**同一个**标量上。汇总这一步天然存在写冲突。

归约是并行计算中最常见的模式之一——求和、求最值、求内积、统计计数，本质都是归约。因此本实验讨论的问题具有普遍性：

> 只要多个线程需要把结果合并到同一个位置，就必然面对同步问题。

选择 π 估算作为载体，是因为它**结构极简、计算量可控、结果可验证**（与 `4*atan(1)` 比对即可）。本实验关心的不是如何高效地计算 π，而是并行时会出什么错、为什么错、以及如何以正确的方式修正。

## 2. 算法：Gregory-Leibniz 级数

$$\pi = 4\sum_{i=0}^{\infty} \frac{(-1)^i}{2i+1}
      = 4\left(1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \cdots\right)$$

该级数收敛极慢（取得 5 位有效数字约需 $10^5$ 项），但其结构简单、项数可调，非常适合作为并发教学的载体。

串行实现：

```c
double sum = 0.0, factor = 1.0;
for (long long i = 0; i < n; ++i, factor = -factor) {
  sum += factor / (2 * i + 1);
}
return 4.0 * sum;
```

### 2.1 并行分解

把 $n$ 项平均分给 $t$ 个线程，各线程计算自己那一段的部分和，再汇总。区间划分方式与实验二相同：

```c
static void compute_range(long my_rank, long long *first, long long *last) {
  long long my_n = n / thread_count;
  *first = my_n * my_rank;
  *last = (my_rank == thread_count - 1) ? n : *first + my_n;
}
```

### 2.2 ⚠️ 一个容易被忽略的细节：符号项的初值

级数中的符号 $(-1)^i$ 是**逐项交替**的，因此每个线程的起始符号取决于其**起始下标的奇偶性**：

```c
static double initial_factor(long long first_i) {
  return (first_i % 2 == 0) ? 1.0 : -1.0;
}
```

若所有线程都从 `factor = 1.0` 开始，结果必然出错。但请注意区分：

> 这是一个**确定性的逻辑错误**——每次运行都错，且错得完全一样。
> 它与本实验要讨论的**数据竞争**是两类不同的缺陷：后者的表现是**每次运行结果都不同**。

把二者混为一谈，会使调试方向完全偏离。判断依据很简单：**重复运行，结果是否稳定**。

## 3. 数据竞争：定义与机器级成因

问题出在这一行：

```c
global_sum += factor / (2 * i + 1);
```

### 3.1 一行 C 语句不是一个原子操作

这行代码看似一步完成，但处理器必须分三步执行：

<!--
| 步骤 | 含义 | x86-64 指令 | AArch64 指令 |
|---|---|---|---|
| 1. 读（Load） | 把 `global_sum` 从内存载入寄存器 | `movsd (%rsi),%xmm1` | `ldr d1,[x0]` |
| 2. 改（Add） | 在寄存器中完成加法 | `addsd %xmm2,%xmm1` | `fadd d1,d1,d2` |
| 3. 写（Store） | 把结果写回内存 | `movsd %xmm1,(%rsi)` | `str d1,[x0]` |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">步骤</th>
      <th style="text-align: left;">含义</th>
      <th style="text-align: left;">x86-64 指令</th>
      <th style="text-align: left;">AArch64 指令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">1. 读（Load）</td>
      <td style="text-align: left;">把 <code>global_sum</code> 从内存载入寄存器</td>
      <td style="text-align: left;"><code>movsd (%rsi),%xmm1</code></td>
      <td style="text-align: left;"><code>ldr d1,[x0]</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">2. 改（Add）</td>
      <td style="text-align: left;">在寄存器中完成加法</td>
      <td style="text-align: left;"><code>addsd %xmm2,%xmm1</code></td>
      <td style="text-align: left;"><code>fadd d1,d1,d2</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">3. 写（Store）</td>
      <td style="text-align: left;">把结果写回内存</td>
      <td style="text-align: left;"><code>movsd %xmm1,(%rsi)</code></td>
      <td style="text-align: left;"><code>str d1,[x0]</code></td>
    </tr>
  </tbody>
</table>

这三条指令并不构成一个不可分割的整体。 一个线程每执行完其中一条，另一个线程都可能插入自己的指令:
> 在单核系统上，操作系统采用抢占式调度：时钟中断到来时，当前线程会在一条指令执行完毕后被暂停，处理器转去执行其他线程，稍后再返回继续。暂停的位置只落在指令边界上，操作系统并不关心一行 C 语句对应的三条指令是否已经执行完。
> 在多核系统上，情形更为直接：两个线程本就运行在不同核心上同时执行，无需任何切换即可发生交错。

于是可能出现如下交错:

<!--
| 时刻 | 线程 A | 线程 B | 内存中的 `global_sum` |
|---|---|---|---|
| $t_1$ | 读 → 寄存器得到 100 | | 100 |
| $t_2$ | | 读 → 寄存器得到 100 | 100 |
| $t_3$ | 加 → 寄存器变为 105 | | 100 |
| $t_4$ | | 加 → 寄存器变为 103 | 100 |
| $t_5$ | 写 → 回写 105 | | **105** |
| $t_6$ | | 写 → 回写 103 | **103** |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">时刻</th>
      <th style="text-align: left;">线程 A</th>
      <th style="text-align: left;">线程 B</th>
      <th style="text-align: left;">内存中的 <code>global_sum</code></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">t₁</td>
      <td style="text-align: left;">读 → 寄存器得到 100</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">100</td>
    </tr>
    <tr>
      <td style="text-align: left;">t₂</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">读 → 寄存器得到 100</td>
      <td style="text-align: left;">100</td>
    </tr>
    <tr>
      <td style="text-align: left;">t₃</td>
      <td style="text-align: left;">加 → 寄存器变为 105</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">100</td>
    </tr>
    <tr>
      <td style="text-align: left;">t₄</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">加 → 寄存器变为 103</td>
      <td style="text-align: left;">100</td>
    </tr>
    <tr>
      <td style="text-align: left;">t₅</td>
      <td style="text-align: left;">写 → 回写 105</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"><strong>105</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">t₆</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;">写 → 回写 103</td>
      <td style="text-align: left;"><strong>103</strong></td>
    </tr>
  </tbody>
</table>

线程 A 的更新被线程 B 覆盖。正确结果应为 108，实际得到 103。这一现象称为**丢失更新**（lost update）。

### 3.2 数据竞争的严格定义

一段程序存在**数据竞争**，当且仅当以下三个条件同时成立：

1. 两个或以上线程**并发访问**同一个内存位置；
2. 其中**至少有一个是写**操作；
3. 这些访问之间**没有任何同步**来强制其先后顺序。

对照实验二逐条检查：`A`、`x` 只读，不满足条件 2；`y` 虽被写入但区间不重叠，不满足条件 1。因此实验二无需加锁。而本实验中 `global_sum` 三个条件全部满足，必然出错。

### 3.3 编译器会放大后果

以上分析是教科书式的推导，它给出的结论是「偶尔丢失一次更新」，误差应当很小。但实际运行中，误差往往达到几个数量级。

原因在于：**C11 标准规定数据竞争属于未定义行为**（undefined behavior）。既然行为未定义，编译器就有权假设「不存在其他线程访问该变量」，进而把 `global_sum` **在整个循环期间保留在寄存器中**，只在循环前读一次、循环后写一次。

其后果是：丢失的不再是某一次加法，而是**某个线程的整段部分和**。

这一论断可以直接验证——第 7 节将用 `objdump` 查看实际生成的机器码。

## 4. 环境准备

下面的单元格检查编译器与硬件环境，并定义本实验统一使用的工具函数。

数据竞争是否显现取决于线程调度：**核心数越多、并发程度越高，竞态越容易暴露**。核心数为 1 时线程只能分时轮转，竞态有可能不出现，甚至碰巧得到正确结果。

In [ ]:
import platform, subprocess, shutil, sys, os, re, glob

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n⚠️  当前仅 1 个核心：竞态可能不显现，加速比亦无从体现，")
    print("    建议在华为鲲鹏多核处理器上运行本实验。")
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### 核心同构性检查

本实验需要比较串行与三个并行版本的耗时。若处理器采用**大小核**架构，串行基准与工作线程可能运行在不同类型的核心上，测得的加速比将失真（详见实验二第 9 节）。

In [ ]:
def detect_core_types():
    """按最高频率给核心分组；返回 {频率(kHz): [核心编号]}，无信息时返回 None。"""
    freqs = {}
    for path in sorted(
        glob.glob("/sys/devices/system/cpu/cpu[0-9]*/cpufreq/cpuinfo_max_freq")
    ):
        cpu = int(path.split("/")[5][3:])
        try:
            freqs[cpu] = int(open(path).read().strip())
        except OSError:
            pass
    if not freqs:
        return None
    groups = {}
    for cpu, f in freqs.items():
        groups.setdefault(f, []).append(cpu)
    return dict(sorted(groups.items(), reverse=True))


def cpu_list(cpus):
    """把核心编号压缩为 taskset 可用的区间表示，如 0-7,12."""
    cpus = sorted(cpus)
    parts, s, prev = [], cpus[0], cpus[0]
    for c in cpus[1:] + [None]:
        if c is not None and c == prev + 1:
            prev = c
            continue
        parts.append(str(s) if s == prev else f"{s}-{prev}")
        s = prev = c
    return ",".join(parts)


groups = detect_core_types()
if groups is None:
    print("未读取到 cpufreq 信息（容器或虚拟机中常见），无法判断核心类型。")
elif len(groups) == 1:
    print(
        f"✅ 全部 {len(next(iter(groups.values())))} 个核心同构，可直接进行性能对比。"
    )
else:
    print(f"⚠️  检测到 {len(groups)} 种不同类型的核心（大小核架构）：")
    for f, cpus in groups.items():
        print(f"    {f/1e6:.2f} GHz : {len(cpus):2d} 个核心  ->  {cpu_list(cpus)}")
    print(f"\n建议绑定到同一类核心后再测量，例如：")
    print(
        f"    taskset -c {cpu_list(next(iter(groups.values())))} ./src_pi/03_pthread_pi 10000000 4"
    )


### 编译与运行工具函数

本实验的编译选项与全章一致；`-lm` 用于链接 `fabs` 与 `atan`。

In [22]:
SRC_DIR = "src_pi"
os.makedirs(SRC_DIR, exist_ok=True)


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    rc, log = sh(cmd)
    if rc == 0:
        print("✅ 编译成功：", cmd)
        if log.strip():
            print(log.strip())
        return out
    print("❌ 编译失败：\n", log)
    return None


def run_bin(out, *args, echo=True):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args], capture_output=True, text=True
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析程序输出的方法对照表，返回 [(方法, 耗时ms, 加速比, 校验)]。"""
    rows = []
    for line in text.splitlines():
        m = re.match(
            r"^(Serial \(baseline\)|Race condition|Fine-grained mutex|"
            r"Coarse-grained mutex)\s+([\d.]+)\s+([\d.]+)x\s+(\w+)\s*$",
            line,
        )
        if m:
            rows.append((m.group(1), float(m.group(2)), float(m.group(3)), m.group(4)))
    return rows


def parse_pi(text, label):
    """从输出中提取某个版本得到的 π 值。"""
    m = re.search(re.escape(label) + r"\s*=\s*([-\d.]+)", text)
    return float(m.group(1)) if m else None


## 5. 三个版本的设计

本实验的程序在**同一次运行中**依次执行串行基准与三个并行版本，使四者面对完全相同的项数与数据，比较才具可比性。

本节先分别剖析三个并行版本的设计意图，第 6 节再给出完整源码并运行。

### 5.1 版本一：无同步（竞态）

```c
void *Thread_sum_race(void *rank) {
  ...
  for (long long i = my_first_i; i < my_last_i; ++i, factor = -factor) {
    global_sum += factor / (2 * i + 1);      // 直接累加到共享变量，无任何保护
  }
  return NULL;
}
```

这是最直接、也最容易写出的版本。它对照第 3.2 节的三个条件全部命中：多个线程并发访问 `global_sum`，其中有写操作，且没有任何同步。

**预期结果**：数值错误，且每次运行的错误程度不同。

### 5.2 版本二：细粒度锁

```c
void *Thread_sum_fine(void *rank) {
  ...
  for (long long i = my_first_i; i < my_last_i; ++i, factor = -factor) {
    pthread_mutex_lock(&mutex);
    global_sum += factor / (2 * i + 1);      // 临界区仅包含这一行
    pthread_mutex_unlock(&mutex);
  }
  return NULL;
}
```

**互斥量**（mutual exclusion）保证任意时刻至多一个线程处于临界区内：

```c
pthread_mutex_t mutex;
pthread_mutex_init(&mutex, NULL);    // 使用前初始化
pthread_mutex_lock(&mutex);          // 进入临界区；若已被占用则阻塞
pthread_mutex_unlock(&mutex);        // 离开临界区
pthread_mutex_destroy(&mutex);       // 用毕销毁
```

**正确性没有问题**：临界区确实保护了那一行读—改—写。

**但代价极高**：
- 每次迭代都要加锁与解锁，而临界区内只有一次浮点除法和一次加法；
- 临界区几乎覆盖了全部计算，各线程实际上是**串行**执行的，还额外付出了锁的开销；
- 锁竞争激烈时，获取失败的线程会被内核挂起并让出处理器，被唤醒时还要承担上下文切换与缓存失效的代价。

**预期结果**：数值正确，但耗时**显著高于串行版本**。

### 5.3 版本三：粗粒度锁

```c
void *Thread_sum_coarse(void *rank) {
  ...
  double my_sum = partial_sum(my_first_i, my_last_i);   // 在私有变量上完成全部计算

  pthread_mutex_lock(&mutex);
  global_sum += my_sum;                                 // 每个线程只加锁一次
  pthread_mutex_unlock(&mutex);
  return NULL;
}
```

**为什么 `my_sum` 不需要保护**：它是函数内的局部变量，位于**线程私有的栈**上。每个线程各有一份，其他线程既看不到也无法访问其地址，不满足数据竞争的条件 1。

**收益**：加锁次数由 $n$ 次降至 $t$ 次（$n$ 通常是 $10^7$ 量级，而 $t$ 只是个位数）。计算部分完全并行，仅最后的汇总是串行的。

> **规则一：最小化临界区。**
> 锁保护的是**数据**，不是**计算**。凡是能在私有变量上完成的工作，一律移出临界区。

这条规则在本章反复出现：实验七的局部部分和、实验八的链表操作统计，用的都是同一手法。

### ⚠️ 关于 `partial_sum` 的说明

细心的读者会注意到，版本三调用了一个 `partial_sum` 函数，而非在函数体内直接书写累加循环。这是**有意为之**：串行基准 `Serial_pi` 调用的是同一个 `partial_sum`。

```c
static double partial_sum(long long first, long long last);

// 串行基准
static double Serial_pi(long long terms) { return 4.0 * partial_sum(0, terms); }

// 版本三
double my_sum = partial_sum(my_first_i, my_last_i);
```

若把同一段循环书写两遍，编译器对二者的优化决策可能不同，测得的差异便不再来自并行本身（该问题在实验二第 5 节已详细讨论）。共用同一个内核函数，可以从根本上排除这一干扰。

## 6. 完整源码与运行

下面用 `%%writefile` 将完整源码写入 `src_pi/pthread_pi_race_condition_and_mutex.c`。程序在一次运行中依次执行串行基准与三个并行版本，并输出对照表。

In [ ]:
%%writefile {SRC_DIR}/pthread_pi_race_condition_and_mutex.c
#include <math.h>
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_THREADS 64

// Shared by all threads.
long thread_count = 0;
long long n = 0;
double global_sum = 0.0;
pthread_mutex_t mutex;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static const char* check_diff(double ref, double test) {
  return fabs(ref - test) < 1e-5 ? "PASS" : "FAIL";
}

// Computes the half-open range [first, last) handled by my_rank.
static void compute_range(long my_rank, long long* first, long long* last) {
  long long my_n = n / thread_count;
  *first = my_n * my_rank;
  *last = (my_rank == thread_count - 1) ? n : *first + my_n;
}

// The alternating sign of the Gregory-Leibniz series depends on the parity of
// the starting index, so each thread must derive its own initial factor.
static double initial_factor(long long first_i) {
  return (first_i % 2 == 0) ? 1.0 : -1.0;
}

// Accumulates terms [first, last) into a private variable. The serial baseline
// and the coarse-grained version both call this function, so the two timings
// compare the same machine code instead of two separately compiled loops.
static double partial_sum(long long first, long long last) {
  double sum = 0.0;
  double factor = initial_factor(first);
  for (long long i = first; i < last; ++i, factor = -factor) {
    sum += factor / (2 * i + 1);
  }
  return sum;
}

static double Serial_pi(long long terms) { return 4.0 * partial_sum(0, terms); }

// Version 1: unsynchronized. global_sum += ... is a read-modify-write that the
// scheduler may interrupt between the three machine instructions, so updates
// are lost and the result varies from run to run.
void* Thread_sum_race(void* rank) {
  long my_rank = (long)rank;
  long long my_first_i, my_last_i;
  compute_range(my_rank, &my_first_i, &my_last_i);
  double factor = initial_factor(my_first_i);

  for (long long i = my_first_i; i < my_last_i; ++i, factor = -factor) {
    global_sum += factor / (2 * i + 1);
  }
  return NULL;
}

// Version 2: fine-grained locking. Correct, but the lock is taken once per
// iteration, so the lock traffic dominates and the threads run serially.
void* Thread_sum_fine(void* rank) {
  long my_rank = (long)rank;
  long long my_first_i, my_last_i;
  compute_range(my_rank, &my_first_i, &my_last_i);
  double factor = initial_factor(my_first_i);

  for (long long i = my_first_i; i < my_last_i; ++i, factor = -factor) {
    pthread_mutex_lock(&mutex);
    global_sum += factor / (2 * i + 1);
    pthread_mutex_unlock(&mutex);
  }
  return NULL;
}

// Version 3: coarse-grained locking. The accumulator lives on the thread stack,
// so it needs no protection; the lock is taken once per thread, not per term.
void* Thread_sum_coarse(void* rank) {
  long my_rank = (long)rank;
  long long my_first_i, my_last_i;
  compute_range(my_rank, &my_first_i, &my_last_i);

  double my_sum = partial_sum(my_first_i, my_last_i);

  pthread_mutex_lock(&mutex);
  global_sum += my_sum;
  pthread_mutex_unlock(&mutex);
  return NULL;
}

// Runs one parallel version and returns its wall time in milliseconds.
static double run_version(void* (*worker)(void*), pthread_t* handles,
                          double* pi_out) {
  global_sum = 0.0;
  double start = get_time_ms();
  for (long i = 0; i < thread_count; ++i) {
    if (pthread_create(&handles[i], NULL, worker, (void*)i) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      exit(1);
    }
  }
  for (long i = 0; i < thread_count; ++i) pthread_join(handles[i], NULL);
  double elapsed = get_time_ms() - start;
  *pi_out = 4.0 * global_sum;
  return elapsed;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <n> <thread_count>\n", argv[0]);
    fprintf(stderr, "  n: number of terms of the Gregory-Leibniz series\n");
    return 1;
  }

  n = strtoll(argv[1], NULL, 10);
  thread_count = strtol(argv[2], NULL, 10);

  if (n <= 0) {
    fprintf(stderr, "Error: n must be positive\n");
    return 1;
  }
  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }

  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }
  pthread_mutex_init(&mutex, NULL);

  printf("Pi Estimation: race condition vs lock granularity\n");
  printf("Terms (n): %lld, Threads: %ld\n\n", n, thread_count);

  double start = get_time_ms();
  double pi_serial = Serial_pi(n);
  double time_serial = get_time_ms() - start;

  double pi_race, pi_fine, pi_coarse;
  double time_race = run_version(Thread_sum_race, thread_handles, &pi_race);
  double time_fine = run_version(Thread_sum_fine, thread_handles, &pi_fine);
  double time_coarse =
      run_version(Thread_sum_coarse, thread_handles, &pi_coarse);

  printf("%-22s %12s %10s %8s\n", "Method", "Time (ms)", "Speedup", "Check");
  printf("%-22s %12.3f %9.2fx %8s\n", "Serial (baseline)", time_serial, 1.0,
         "REF");
  printf("%-22s %12.3f %9.2fx %8s\n", "Race condition", time_race,
         time_serial / time_race, check_diff(pi_serial, pi_race));
  printf("%-22s %12.3f %9.2fx %8s\n", "Fine-grained mutex", time_fine,
         time_serial / time_fine, check_diff(pi_serial, pi_fine));
  printf("%-22s %12.3f %9.2fx %8s\n", "Coarse-grained mutex", time_coarse,
         time_serial / time_coarse, check_diff(pi_serial, pi_coarse));

  printf("\nPi values:\n");
  printf("  Reference (4*atan(1)) = %.15f\n", 4.0 * atan(1.0));
  printf("  Serial                = %.15f\n", pi_serial);
  printf("  Race condition        = %.15f\n", pi_race);
  printf("  Fine-grained mutex    = %.15f\n", pi_fine);
  printf("  Coarse-grained mutex  = %.15f\n", pi_coarse);

  pthread_mutex_destroy(&mutex);
  free(thread_handles);
  return 0;
}

In [ ]:
pi_bin = compile_c(
    f"{SRC_DIR}/pthread_pi_race_condition_and_mutex.c", f"{SRC_DIR}/pthread_pi"
)
print()

N_TERMS = 10000000
NT = max(2, min(4, os.cpu_count()))
out = run_bin(pi_bin, N_TERMS, NT)


### 首轮观察

请重点关注三处：

1. **`Race condition` 一行的 `Check` 为 `FAIL`**，且其 π 值与参考值相差极大——不是「略有偏差」，而是完全错误。
2. **`Fine-grained mutex` 结果正确，但耗时远高于串行**。加锁使程序退化为串行执行，还额外付出了锁的开销。
3. **`Coarse-grained mutex` 既正确又快速**，其耗时与串行基准相当或更低（取决于可用核心数）。

竞态版的误差为何如此之大？下一节用机器码给出答案。

## 7. 机器级验证：编译器如何放大数据竞争

第 3.3 节提出一个论断：编译器有权把 `global_sum` 在整个循环期间保留在寄存器中。下面直接查看生成的机器码予以验证。

判断方法：提取循环体（由向后跳转指令界定），检查其中**是否存在经由寄存器间接寻址的内存访问**。若循环体内没有对 `global_sum` 的读写，即说明该变量已被提升到寄存器。

In [ ]:
def loop_body(binary, func):
    """提取函数中最内层循环的指令，依据第一条向后跳转判定。"""
    rc, asm = sh(f"objdump -d {binary} --disassemble={func}")
    lines = [l.rstrip() for l in asm.split("\n") if re.match(r"^\s+[0-9a-f]+:", l)]
    addr = lambda l: int(re.match(r"^\s+([0-9a-f]+):", l).group(1), 16)
    for i, l in enumerate(lines):
        m = re.search(
            r"\b(?:jne|jnz|je|jl|jle|jg|jge|jb|jae|bne|beq|cbnz|cbz|b\.\w+)"
            r"\s+([0-9a-f]+)",
            l,
        )
        if m and int(m.group(1), 16) < addr(l):
            t = int(m.group(1), 16)
            return [x for x in lines if t <= addr(x) <= addr(l)]
    return []


def mem_ops(lines):
    """统计经由寄存器间接寻址的内存访问，排除 %rip 相对寻址的常量池。"""
    n = 0
    for l in lines:
        txt = l.split("\t")[-1]
        if "(%rip)" in txt:
            continue
        if re.search(r"\(%r[a-z0-9]+\)", txt) or re.search(r"\[[xw][0-9]+", txt):
            n += 1
    return n


for fn in ["Thread_sum_race", "Thread_sum_coarse"]:
    body = loop_body(pi_bin, fn)
    print(f"═══ {fn} ═══  循环体 {len(body)} 条指令，其中内存访问 {mem_ops(body)} 处")
    for l in body:
        print("    " + re.sub(r"^\s+", "", l)[:76])
    print()


### 💡 验证结果的解读

若观察到**竞态版本的循环体内没有任何内存访问**，则第 3.3 节的论断得到证实：编译器把 `global_sum` 载入寄存器，在整个循环中于寄存器内累加，仅在循环**结束后**写回内存一次。

更值得注意的是：**竞态版与粗粒度锁版的循环体往往完全相同**。这意味着编译器实际上把「直接累加到共享变量」自动改写成了「先在寄存器中累加、最后写一次」——这恰恰就是粗粒度锁版本刻意采取的策略，**唯独少了那一把互斥量**。

由此可以解释第 6 节的两个现象：

<!--
| 现象 | 原因 |
|---|---|
| 竞态版与粗粒度锁版**耗时几乎相同** | 二者的计算循环是同一段代码 |
| 竞态版的结果**错得极其离谱** | 各线程在循环结束时对 `global_sum` 各写一次，互相覆盖；最终只有最后一个写入者的部分和得以保留，其余线程的全部工作尽数丢失 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">现象</th>
      <th style="text-align: left;">原因</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">竞态版与粗粒度锁版<strong>耗时几乎相同</strong></td>
      <td style="text-align: left;">二者的计算循环是同一段代码</td>
    </tr>
    <tr>
      <td style="text-align: left;">竞态版的结果<strong>错得极其离谱</strong></td>
      <td style="text-align: left;">各线程在循环结束时对 <code>global_sum</code> 各写一次，互相覆盖；最终只有最后一个写入者的部分和得以保留，其余线程的全部工作尽数丢失</td>
    </tr>
  </tbody>
</table>

因此，丢失的并非「某一次加法」，而是「某个线程的整段部分和」。这也解释了为何误差可以达到几个数量级。

> **结论**：未同步的共享写，其行为不仅取决于硬件调度，也取决于编译器优化。
> 由于数据竞争在 C11 标准中属于未定义行为，编译器可以做出任何变换——包括让错误变得远比直觉严重。
>
> 这一点也说明：**试图通过「推测调度时序」来论证一段竞态代码「大概率没问题」，是徒劳的。**

## 8. 竞态的不可复现性

数据竞争最危险的特征不是「会出错」，而是「**错得不稳定**」。下面连续运行 10 次，观察竞态版本每次的结果。

In [ ]:
import math

REF = 4.0 * math.atan(1.0)
vals, serial_ref = [], None
for i in range(10):
    o = run_bin(pi_bin, N_TERMS, NT, echo=False)
    v = parse_pi(o, "Race condition")
    s = parse_pi(o, "Serial")
    serial_ref = s
    vals.append(v)
    print(f"第 {i+1:2d} 次：竞态结果 = {v:.12f}   与串行的偏差 = {abs(v - s):.3e}")

uniq = len(set(vals))
print(f"\n10 次运行共得到 {uniq} 个不同的结果")
print(f"最小值 = {min(vals):.12f}")
print(f"最大值 = {max(vals):.12f}")
print(f"参考值 = {REF:.12f}（串行结果 {serial_ref:.12f}）")

if uniq == 1:
    print(f"\n[说明] 本机核心数 = {os.cpu_count()}。核心数较少时线程往往依次执行完毕，")
    print("竞态的交错窗口不易命中，结果可能稳定，甚至碰巧正确。")
    print("这恰恰是数据竞争最危险之处：在开发机上测不出来，部署到多核环境即暴露。")
    print("请在鲲鹏多核平台上重跑本单元。")


### 💡 为何并发缺陷难以排查

一个存在数据竞争的程序，通常具有以下特征：

- 在开发机上运行上百次可能都正确；
- 换到核心数更多的机器上便开始出错；
- 加入 `printf` 调试语句后，因时序被改变，错误又消失了（此类缺陷因而被称为 **Heisenbug**）；
- 用 `-O0` 编译能通过，`-O3` 就出错（原因见第 7 节）。

> **程序在某次运行中碰巧正确，不等于程序逻辑正确。**

这正是本章反复强调「先保证正确，再谈优化」的原因。判断并行程序是否正确，依据必须是**对照数据竞争的定义所做的推理**，而不是「多跑几次没出问题」。

## 9. 锁粒度对性能的影响

下面把四个版本的耗时与加速比汇总为图表。注意：**加速比只对结果正确的版本才有意义**，因此图中会标出各版本的校验状态。

In [ ]:
import matplotlib.pyplot as plt

rows = parse_table(run_bin(pi_bin, N_TERMS, NT, echo=False))
cn = {
    "Serial (baseline)": "串行基准",
    "Race condition": "竞态（无锁）",
    "Fine-grained mutex": "细粒度锁",
    "Coarse-grained mutex": "粗粒度锁",
}

print(f"{'版本':<14}{'耗时(ms)':>12}{'加速比':>10}{'正确性':>10}")
print("-" * 48)
for name, t, s, chk in rows:
    print(f"{cn[name]:<14}{t:>12.3f}{s:>9.2f}x{chk:>10}")

# 图中一律使用英文标签，避免依赖中文字体
en = {
    "Serial (baseline)": "Serial baseline",
    "Race condition": "Race (no lock)",
    "Fine-grained mutex": "Fine-grained mutex",
    "Coarse-grained mutex": "Coarse-grained mutex",
}
labels = [en[r[0]] for r in rows]
speeds = [r[2] for r in rows]
checks = [r[3] for r in rows]
# 灰=基准，红=结果错误，橙=正确但低效，绿=正确且高效
colors = []
for name, t, s, chk in rows:
    if chk == "REF":
        colors.append("#9aa0a6")
    elif chk == "FAIL":
        colors.append("#C7000B")
    elif s < 1.0:
        colors.append("#E8833A")
    else:
        colors.append("#2E7D32")

fig, ax = plt.subplots(figsize=(8.5, 4.4))
bars = ax.bar(labels, speeds, color=colors)
ax.axhline(1.0, ls="--", c="gray", lw=1.2, label="Serial baseline")
for b, s, chk in zip(bars, speeds, checks):
    tag = f"{s:.2f}x" + ("\nwrong result" if chk == "FAIL" else "")
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        tag,
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Speedup vs serial")
ax.set_title(f"Pi estimation n={N_TERMS:,}, {NT} threads: "
             f"effect of lock granularity ({os.cpu_count()} cores)")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fine = next(r for r in rows if r[0] == "Fine-grained mutex")
coarse = next(r for r in rows if r[0] == "Coarse-grained mutex")
print(
    f"\n粗粒度锁比细粒度锁快 {fine[1] / coarse[1]:.1f} 倍，"
    f"而二者的计算量完全相同，差别仅在于加锁的次数与位置。"
)


### 结果解读

**① 细粒度锁往往慢于串行。** 每次迭代都要经历「加锁 → 一次浮点除法与加法 → 解锁」。锁竞争激烈时，获取失败的线程被内核挂起并让出处理器，被唤醒时还需承担上下文切换与缓存失效的代价。这些开销远大于它所保护的那点计算。

> **加锁本身并不慢，慢的是加错了位置。**

**② 粗粒度锁的加速比接近线程数。** 计算部分完全并行，锁仅在汇总时使用一次。其耗时与串行基准的比值，才是本实验真正有意义的加速比。

**③ 竞态版本可能很快，但结果是错的。** 把错误的结果算得再快也没有意义。表中的 `Check` 一列比 `Speedup` 一列重要。

### 关于本机结果

若加速比明显低于线程数，请核对第 4 节输出的核心数与核心类型。在单核环境下，三个并行版本都无法获得真正的并行加速，本节图表仅能反映锁开销的相对大小；粗粒度锁与串行基准耗时相当，即为预期结果。

## 10. 结果分析

本实验建立了三项贯穿全章的认识：

**① 归约必然带来写冲突，必须同步。** 只要多个线程把结果合并到同一位置，数据竞争的三个条件就会同时成立。判断是否需要同步，依据是对照定义所做的推理，而非运行结果是否碰巧正确。

**② 未定义行为的后果不可推测。** 编译器有权把未同步的共享变量提升到寄存器，使「丢失一次更新」升级为「丢失整段部分和」。因此不能用「推测调度时序」来论证竞态代码的安全性。

**③ 临界区的大小直接决定并行程序的性能。** 同样使用互斥量，细粒度锁使程序退化为串行并额外付出锁开销；粗粒度锁则让计算完全并行。

### 🎓 结论

三个版本的**计算量完全相同**，差别仅在于同步的方式与位置，性能却可以相差一个数量级，正确性更是判若云泥。由此可以提炼出并行程序设计的基本次序：

> **第一步：用数据竞争的定义判断哪些位置需要同步。**
> **第二步：用最小的临界区实现该同步。**

顺序不能颠倒。先追求性能而忽略正确性，得到的是一个快速产出错误结果的程序；而正确性一旦保证，性能问题总有优化的余地。

实验四将把这一次序用在一个更复杂的场景上：当一个线程需要同时持有多把锁时，即使每一处同步都正确，程序仍可能陷入死锁——所有线程都在等待，却谁也无法推进。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察行为与性能的变化：

1. 将线程数依次设为 1、2、4、8、16，记录三个并行版本的耗时，绘制加速比曲线，并解释细粒度锁曲线的形状。
2. 删去 `initial_factor` 函数，令所有线程都从 `factor = 1.0` 开始，运行多次。记录结果是否稳定，并说明该错误与数据竞争在表现上有何本质区别。
3. 在细粒度锁版本中，把 `pthread_mutex_lock/unlock` 换成 C11 原子操作（需将 `global_sum` 改为定点整型并使用 `atomic_fetch_add`），对比性能，说明原子操作与互斥量的开销差异。
4. 将粗粒度锁版本的 `my_sum` 由局部变量改为全局数组 `double partial[MAX_THREADS]`，令每个线程写 `partial[my_rank]`，测量性能变化并解释原因（提示：参见实验九）。
5. 把项数 `n` 由 $10^7$ 依次降到 $10^6$、$10^5$、$10^4$，观察粗粒度锁版本的加速比如何变化，找出并行不再有利可图的规模临界点。

## 12. 🤔 思考题

- 竞态版本每次运行的结果都不同，但都远小于正确值。请结合第 7 节的机器码分析，解释这一系统性偏差的方向。
- 本实验的临界区只有 `global_sum += my_sum;` 一行。能否把这一行也去掉，改为「每个线程写 `partial[my_rank]`，主线程 `join` 后再汇总」？这样做还需要锁吗？与粗粒度锁相比孰优孰劣？
- 假设有人提出「在细粒度锁版本外面再套一层锁，性能就会变好」。请指出这一说法错在何处。
- 浮点加法不满足结合律，因此串行版与粗粒度锁版的结果在最低位上可能不同。本实验用 `1e-5` 的容差判定通过。这一容差是否可能掩盖真正的竞态错误？如何设计既能容忍浮点误差、又能捕捉丢失更新的判据？
- 教材指出「数据竞争是未定义行为，编译器可以假设它不存在」。若编译器不作此假设，将会失去哪些常见的优化能力？请举例说明。

## 13. 小结与后续

本实验通过同一问题的三种同步策略，完成了从「并行出错」到「并行正确且高效」的完整过程：

<!--
| 版本 | 同步方式 | 正确性 | 性能 | 新增知识点 |
|---|---|---|---|---|
| **版本一** | 无 | **错误** | 快（但无意义） | 数据竞争的定义与机器级成因、未定义行为 |
| **版本二** | 每次迭代加锁 | 正确 | **劣于串行** | 互斥量 API、锁开销的来源 |
| **版本三** | 汇总时加锁一次 | 正确 | **接近线性加速** | 最小化临界区、线程私有栈变量无需保护 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">同步方式</th>
      <th style="text-align: left;">正确性</th>
      <th style="text-align: left;">性能</th>
      <th style="text-align: left;">新增知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>版本一</strong></td>
      <td style="text-align: left;">无</td>
      <td style="text-align: left;"><strong>错误</strong></td>
      <td style="text-align: left;">快（但无意义）</td>
      <td style="text-align: left;">数据竞争的定义与机器级成因、未定义行为</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>版本二</strong></td>
      <td style="text-align: left;">每次迭代加锁</td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;"><strong>劣于串行</strong></td>
      <td style="text-align: left;">互斥量 API、锁开销的来源</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>版本三</strong></td>
      <td style="text-align: left;">汇总时加锁一次</td>
      <td style="text-align: left;">正确</td>
      <td style="text-align: left;"><strong>接近线性加速</strong></td>
      <td style="text-align: left;">最小化临界区、线程私有栈变量无需保护</td>
    </tr>
  </tbody>
</table>

本实验确立的规则将在后续实验中反复应用：

1. **最小化临界区**——锁保护的是数据而非计算（实验七、实验八）；
2. **正确性优先于性能**——先用定义判断是否需要同步，再考虑如何高效实现（全章）；
3. **不可依据「多跑几次没出问题」判断并发程序的正确性**（全章）。

➡️ **后续内容：实验四 哲学家就餐：死锁、活锁与资源分级**。本实验中，每个线程在任一时刻只需要持有一把锁。实验四把条件改为：每个线程必须**同时持有两把**锁才能工作。仅此一个变化，就足以引出并发编程中最著名的两类故障——**死锁与活锁**。

至此互斥量这一工具的能力与边界才算完整：本实验说明了它能解决什么，实验四将说明它用不好会造成什么。二者合起来，构成对互斥量的完整认识。